In [2]:
import pandas as pd
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

In [3]:
import pandas as pd
import re
from pathlib import Path

# Load cleaned county-level datasets from this workspace
base_dir = Path(r"C:/Users/rebec/Documents/BigData/Final Project/Data/Cleaned Data")

population = pd.read_csv(base_dir / "nc_population_data.csv")
geo = pd.read_csv(base_dir / "nc_geo_survivability_by_county.csv")
health = pd.read_csv(base_dir / "nc_county_health_scores.csv")
education = pd.read_csv(base_dir / "nc_education_awareness_scores.csv")
mobility = pd.read_csv(base_dir / "nc_mobility_with_counties.csv")
social = pd.read_csv(base_dir / "nc_soc_com_scores.csv")
prep = pd.read_csv(base_dir / "nc_county_infrastructure_scores.csv")
pop_density = pd.read_csv(base_dir / "pop_density.csv")


def normalize_county_name(value):
    """Normalize county text to a stable join key across files."""
    if pd.isna(value):
        return None
    text = str(value).strip().upper()

    # Remove common state/county suffixes and punctuation that differ by source file.
    text = re.sub(r",?\s*NORTH\s+CAROLINA\b", "", text)
    text = re.sub(r"\bCOUNTY\b", "", text)
    text = re.sub(r"[^A-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None


def ensure_county_columns(df):
    """Ensure each dataframe has county and county_key columns.

    Supports files that already contain county_key (for example pop_density.csv).
    """
    out = df.copy()

    if "county_key" in out.columns:
        out["county_key"] = out["county_key"].map(normalize_county_name)
        if "county" not in out.columns:
            out["county"] = out["county_key"]
        return out

    # Detect whichever county column exists in each file.
    candidates = ["county", "County", "county_name", "CountyName", "County Name", "NAME", "County_Name"]
    county_col = next((c for c in candidates if c in out.columns), None)
    if county_col is None:
        raise ValueError(f"No county column found. Available columns: {list(out.columns)}")

    out["county"] = out[county_col].astype(str).str.strip()
    out["county_key"] = out["county"].map(normalize_county_name)
    return out


def drop_county_name_columns(df):
    # Keep only canonical county fields on merge output.
    to_drop = [c for c in ["County", "CountyName", "county_name", "County Name", "County_Name", "NAME"] if c in df.columns]
    return df.drop(columns=to_drop, errors="ignore")


# Standardize all source files
population = drop_county_name_columns(ensure_county_columns(population))
geo = drop_county_name_columns(ensure_county_columns(geo))
health = drop_county_name_columns(ensure_county_columns(health))
education = drop_county_name_columns(ensure_county_columns(education))
mobility = drop_county_name_columns(ensure_county_columns(mobility))
social = drop_county_name_columns(ensure_county_columns(social))
prep = drop_county_name_columns(ensure_county_columns(prep))
pop_density = drop_county_name_columns(ensure_county_columns(pop_density))

# Base frame from population to guarantee one row per county
base_cols = ["county", "county_key"] + [c for c in population.columns if c not in ["county", "county_key"]]
df = population[base_cols].copy()

# Merge each source and report matched counties safely
sources = [
    ("geo", geo),
    ("health", health),
    ("education", education),
    ("mobility", mobility),
    ("social", social),
    ("prep", prep),
    ("pop_density", pop_density),
]

for name, right_df in sources:
    right_clean = right_df.drop(columns=[c for c in ["county", "County", "CountyName", "county_name", "County Name", "County_Name", "NAME"] if c in right_df.columns], errors="ignore")

    before = len(df)
    merged = df.merge(right_clean, on="county_key", how="left", indicator=True)
    matched = int((merged["_merge"] == "both").sum())
    print(f"{name}: matched {matched}/{before} counties")

    df = merged.drop(columns=["_merge"])

print("\nFinal merged shape:", df.shape)
print("Counties with any missing values:", int(df.isna().any(axis=1).sum()))
print("\nExample dataframe columns:")
print(df.columns.tolist())
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/rebec/Documents/BigData/Final Project/Data/Cleaned Data/nc_population_data.csv'

In [ ]:
# Keep a stable modeling subset whether columns are pre-rename or post-rename.
column_aliases = {
    "county_key": ["county_key"],
    "population": ["population"],
    "geo_score": ["mean_survivability", "geo_score"],
    "health_score": ["score_H", "health_score"],
    "edu_score": ["edu_awareness_score", "edu_score"],
    "mobility_score": ["score_M", "mobility_score"],
    "social_score": ["final_score", "social_score"],
    "prep_score": ["score_I", "prep_score"],
    "safety_density_score": ["safety_density_score"],
}

selected = pd.DataFrame(index=df.index)
missing_required = []

for target, candidates in column_aliases.items():
    source = next((c for c in candidates if c in df.columns), None)
    if source is None:
        if target == "safety_density_score":
            # Optional in case pop density file is not merged yet.
            selected[target] = pd.NA
        else:
            missing_required.append(target)
    else:
        selected[target] = df[source]

if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}. Available: {list(df.columns)}")

df = selected
display(df.head())

In [ ]:
# Example dataframe columns:
# county_key, population, geo_score, health_score, edu_score, mobility_score, social_score, prep_score

# -----------------------------
# 0. Regional imputation for missing scores
# -----------------------------
MOUNTAIN_COUNTIES = {
    "ALLEGHANY", "ASHE", "AVERY", "BUNCOMBE", "BURKE", "CALDWELL", "CHEROKEE", "CLAY", "GRAHAM",
    "HAYWOOD", "HENDERSON", "JACKSON", "MACON", "MADISON", "MCDOWELL", "MITCHELL", "SWAIN",
    "TRANSYLVANIA", "WATAUGA", "WILKES", "YANCEY"
}

COASTAL_COUNTIES = {
    "BEAUFORT", "BERTIE", "BLADEN", "BRUNSWICK", "CAMDEN", "CARTERET", "CHOWAN", "COLUMBUS",
    "CRAVEN", "CURRITUCK", "DARE", "DUPLIN", "EDGECOMBE", "GATES", "GREENE", "HALIFAX", "HERTFORD",
    "HYDE", "JONES", "LENOIR", "MARTIN", "NASH", "NEW HANOVER", "NORTHAMPTON", "ONSLOW", "PAMLICO",
    "PASQUOTANK", "PENDER", "PERQUIMANS", "PITT", "ROBESON", "SAMPSON", "TYRRELL", "WASHINGTON",
    "WAYNE", "WILSON"
}


def county_region(county_key):
    if pd.isna(county_key):
        return "piedmont"
    if county_key in MOUNTAIN_COUNTIES:
        return "mountain"
    if county_key in COASTAL_COUNTIES:
        return "coastal"
    return "piedmont"


def fill_missing_scores_by_region(df):
    out = df.copy()
    out["region"] = out["county_key"].map(county_region)

    score_cols = ["geo_score", "health_score", "edu_score", "mobility_score", "social_score", "prep_score"]

    before_missing = out[score_cols].isna().sum()

    for col in score_cols:
        # First pass: fill with region mean.
        out[col] = out[col].fillna(out.groupby("region")[col].transform("mean"))
        # Fallback: if an entire region is NaN for this column, use statewide mean.
        out[col] = out[col].fillna(out[col].mean())

    after_missing = out[score_cols].isna().sum()

    print("Missing scores before regional fill:")
    print(before_missing)
    print("\nMissing scores after regional fill:")
    print(after_missing)

    return out


# -----------------------------
# 1. Build vulnerability and defense scores
# -----------------------------
def compute_scores(df):
    out = df.copy()

    # out["V"] = (
    #     0.30 * out["geo_score"] +
    #     0.20 * out["edu_score"] +
    #     0.30 * out["mobility_score"] +
    #     0.20 * out["prep_score"]
    # )

    out ["V"] = (
        0.20 * df["geo_score"] +
        0.20 * df["edu_score"] +
        0.20 * df["mobility_score"] +
        0.15 * df["prep_score"] +
        0.25 * df["safety_density_score"]
    )

    out["D"] = (
        0.30 * out["health_score"] +
        0.20 * out["edu_score"] +
        0.40 * out["social_score"] +
        0.10 * out["prep_score"]
    )

    return out

# -----------------------------
# 2. County-specific beta and kappa
# Beta and kappa are based on Last of Us scores provided by Curtis
# Can change beta and kappa values if needed
# -----------------------------
def compute_parameters(df, beta0=0.009, kappa0=0.0045, a=0.5, b=0.5):
    out = df.copy()

    out["beta"] = beta0 * (1 - a * out["V"])
    out["kappa"] = kappa0 * (1 + b * out["D"])

    # Prevent negative or absurd values
    out["beta"] = out["beta"].clip(lower=beta0 * 0.2)
    out["kappa"] = out["kappa"].clip(lower=kappa0 * 0.2, upper=kappa0 * 2.0)

    return out

# -----------------------------
# 3. SZR system
# -----------------------------
def szr_model(t, y, beta, kappa):
    S, Z, R = y
    dSdt = -beta * S * Z
    dZdt = (beta - kappa) * S * Z
    dRdt = kappa * S * Z
    return [dSdt, dZdt, dRdt]

# -----------------------------
# 4. Simulate one county
# -----------------------------
def simulate_county(population, beta, kappa, z0=1, days=30):
    S0 = population - z0
    Z0 = z0
    R0 = 0

    t_eval = np.linspace(0, days, 150)

    sol = solve_ivp(
        szr_model,
        t_span=(0, days),
        y0=[S0, Z0, R0],
        t_eval=t_eval,
        args=(beta, kappa)
    )

    return sol

# -----------------------------
# 5. Run for all counties
# -----------------------------
def run_simulations(df, days=60):
    results = []

    for _, row in df.iterrows():
        if pd.isna(row["population"]) or pd.isna(row["beta"]) or pd.isna(row["kappa"]):
            continue

        sol = simulate_county(
            population=row["population"],
            beta=row["beta"],
            kappa=row["kappa"],
            z0=1,
            days=days
        )

        S_final = sol.y[0, -1]
        Z_final = sol.y[1, -1]
        R_final = sol.y[2, -1]
        survival_fraction = S_final / row["population"]

        results.append({
            "county_key": row["county_key"],
            "region": row["region"],
            "beta": row["beta"],
            "kappa": row["kappa"],
            "survival_fraction": survival_fraction,
            "final_susceptible": S_final,
            "final_zombies": Z_final,
            "final_removed": R_final,
            "peak_zombies": sol.y[1].max()
        })

    return pd.DataFrame(results)

# Example usage:
df = fill_missing_scores_by_region(df)
df = compute_scores(df)
df = compute_parameters(df)
results = run_simulations(df)
display(results.sort_values("survival_fraction", ascending=False))

In [ ]:
# -----------------------------
# 1. Build static survivability index (NO V, NO D)
# -----------------------------
def compute_survivability_index(df):
    out = df.copy()

    # Higher = better survivability
    out["survivability_raw"] = (
        0.15 * out["geo_score"] +
        0.20 * out["health_score"] +
        0.10 * out["edu_score"] +
        0.15 * out["mobility_score"] +
        0.25 * out["social_score"] +
        0.15 * out["prep_score"]
    )

    # Normalize to 0–1
    min_val = out["survivability_raw"].min()
    max_val = out["survivability_raw"].max()

    out["survivability_index"] = (
        (out["survivability_raw"] - min_val) /
        (max_val - min_val)
    )

    return out

def compute_parameters(df, beta0=0.009, kappa0=0.0045, a=0.5, b=0.5):
    out = df.copy()

    out["beta"] = beta0 * (1 - a * out["V"])
    out["kappa"] = kappa0 * (1 + b * out["D"])

    # Prevent negative or absurd values
    out["beta"] = out["beta"].clip(lower=beta0 * 0.2)
    out["kappa"] = out["kappa"].clip(lower=kappa0 * 0.2, upper=kappa0 * 2.0)

    return out

# -----------------------------
# 3. SZR system
# -----------------------------
def szr_model(t, y, beta, kappa):
    S, Z, R = y
    dSdt = -beta * S * Z
    dZdt = (beta - kappa) * S * Z
    dRdt = kappa * S * Z
    return [dSdt, dZdt, dRdt]

# -----------------------------
# 4. Simulate one county
# -----------------------------
def simulate_county(population, beta, kappa, z0=1, days=30):
    S0 = population - z0
    Z0 = z0
    R0 = 0

    t_eval = np.linspace(0, days, 150)

    sol = solve_ivp(
        szr_model,
        t_span=(0, days),
        y0=[S0, Z0, R0],
        t_eval=t_eval,
        args=(beta, kappa)
    )

    return sol

# -----------------------------
# 5. Run for all counties
# -----------------------------
def run_simulations(df, days=30):
    results = []

    for _, row in df.iterrows():
        if pd.isna(row["population"]) or pd.isna(row["beta"]) or pd.isna(row["kappa"]):
            continue

        sol = simulate_county(
            population=row["population"],
            beta=row["beta"],
            kappa=row["kappa"],
            z0=1,
            days=days
        )

        S_final = sol.y[0, -1]
        Z_final = sol.y[1, -1]
        R_final = sol.y[2, -1]
        survival_fraction = S_final / row["population"]

        results.append({
            "county_key": row["county_key"],
            "region": row["region"],
            "beta": row["beta"],
            "kappa": row["kappa"],
            "survival_fraction": survival_fraction,
            "final_susceptible": S_final,
            "final_zombies": Z_final,
            "final_removed": R_final,
            "peak_zombies": sol.y[1].max()
        })

    return pd.DataFrame(results)

# Example usage:
df = fill_missing_scores_by_region(df)
df = compute_scores(df)
df = compute_parameters(df)
results = run_simulations(df)
display(results.sort_values("survival_fraction", ascending=False))

In [ ]:
import os

# Create the processed directory if it doesn't exist
os.makedirs("data/processed", exist_ok=True)

# Export the merged DataFrame — this is what integrate_real_hsi.py reads
df.to_csv("data/processed/merged_county_df.csv", index=False)

print(f"Exported {len(df)} counties × {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")